# p4a 执行轨迹观测

前缀层面的问题已在 `01_session_classes.ipynb` 完成实验。

本笔记选择 961 份 sessions ，观察和检测 agent 实际走出来的步骤有多少是共通的、从哪里开始分叉。

观测集是那 961 份。选它的理由见 `docs/experiments/e01-p4a-trajectory.md` §1.3：组内前缀已构造性同质，观察到的分叉可以归因于轨迹本身。

## 1. 筛出观测集

判据是三元组 `(工具Δ, 目录树, delivery)`，取 `(-3882, fb389653, pointer)`，再叠加 s0 的纳入过滤。

In [ ]:
import nbio
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
nbio.banner()

In [ ]:
# 观测集判据。改这里就等于换观测集，其余 cell 不用动。
GROUP = {
    "tools_tok_rel_ref": -3882,      # 轴 A 工具配置
    "axis_tree": "fb389653",         # 轴 D 项目目录树
    "delivery": "pointer",           # 轴 E 投递形态
    "harness_version": "V2",         # 轴 B harness 版本
}

w = nbio.wide()                                   # s0 × s0b × s2 × s3 × s4
sel = w.included.copy()
for k, v in GROUP.items():
    sel &= w[k] == v
g = w[sel].copy()
g["day"] = pd.to_datetime(g.created_at).dt.tz_convert("Asia/Shanghai").dt.date

assert len(g) == 961, f"观测集份数不是 961 而是 {len(g)}，判据或产物变了"
assert g.sysprompt_chars.nunique() == 1, "组内 systemPrompt 长度不唯一"
assert (g.family == "extract").all(), "组内混入了非 extract"

print(f"观测集 {len(g)} 份 | {g.paper_id.nunique()} 篇论文 | {g.day.min()} .. {g.day.max()}")
print(f"sysprompt_chars {g.sysprompt_chars.iloc[0]} | 累计 prefill {g.sum_input.sum():,.0f} tok "
      f"（占纳入集 {g.sum_input.sum() / w[w.included].sum_input.sum():.1%}）")

筛完先自查同质性：组内首步 prompt 的跨度就是这一组"前缀有多齐"的直接度量。

In [ ]:
fs = g.first_step_input
print(f"首步 prompt token: min {fs.min()} max {fs.max()}，跨度 {fs.max() - fs.min()} tok")
print(f"时间戳之后被切断的字符 p50: {g.poisoned_tail_chars.median():.0f}")
print()
print("组内仍在变的列（互异 > 1）：")
vary = {c: g[c].nunique() for c in
        ["axis_harness", "axis_tree", "axis_agents", "axis_skills", "axis_timestamp",
         "sysprompt_chars", "tools_tok_rel_ref", "delivery", "pointer_layout", "pid_year"]
        if g[c].nunique() > 1}
print(vary)

只有时间戳在变，其余各轴组内为常数。这正是 §1.3 说的"L2 按构造成立"。

In [ ]:
desc = (g[["n_steps", "n_tools", "peak_input", "sum_input", "amplification",
           "n_external", "n_bash", "n_read", "n_edit", "n_todo", "n_injections"]]
        .describe(percentiles=[.1, .25, .5, .75, .9]).T
        .drop(columns=["count"]).round(1))
desc